# Validation Module Demo
This notebook demonstrates the usage of the validation tools:
- **Circular Block Bootstrap** for VaR Confidence Intervals.
- **Kupiec POF Test** for backtesting VaR models.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from validation.bootstrap import calculate_historical_var, circular_block_bootstrap
from validation.backtest import kupiec_pof_test

print("Imports successful.")

Imports successful.


In [2]:
# Load Data
data_path = '../data/btc_log_returns.csv'
df = pd.read_csv(data_path, parse_dates=['Date'], index_col='Date')

print(f"Data loaded: {len(df)} observations.")
df.head()

Data loaded: 1856 observations.


,Log_Return
Date,
2019-01-02,0.025657
2019-01-03,-0.027422
2019-01-04,0.005452
2019-01-05,-0.003251
2019-01-06,0.058447


In [6]:
# Calculate Historical VaR (5%)
returns = df['Log_Return'].values
alpha = 0.05

var_95 = calculate_historical_var(returns, alpha)
print(f"Historical VaR (95%): {var_95:.6f}")

# Bootstrap Confidence Interval for VaR
# We use a block size. A common heuristic is N^(1/3) or similar, let's try 20 days.
block_size = 20
lower, observed, upper = circular_block_bootstrap(
    returns, 
    lambda x: calculate_historical_var(x, alpha), 
    block_size=block_size,
    n_bootstrap=1000,
    random_state=42
)

print(f"Bootstrap 95% CI for VaR: [{lower:.6f}, {upper:.6f}]")

Historical VaR (95%): -0.051460
Bootstrap 95% CI for VaR: [-0.059224, -0.046418]
Bootstrap 95% CI for VaR: [-0.059224, -0.046418]


In [7]:
# Kupiec POF Test
# For demonstration, let's assume our model predicted the Historical VaR constant over time.
# In a real scenario, 'var_estimates' would be a time series of predictions.

var_estimates = np.full_like(returns, var_95)

results = kupiec_pof_test(returns, var_estimates, alpha=alpha)

print("Kupiec POF Test Results:")
for k, v in results.items():
    print(f"{k}: {v}")

if results['decision'] == 1:
    print("\nResult: Reject H0 (Model is inaccurate)")
else:
    print("\nResult: Fail to reject H0 (Model is acceptable)")

Kupiec POF Test Results:
LR_POF: 0.0004534120483205691
p_value: 0.9830115494041615
failures: 93
expected_failures: 92.80000000000001
observed_rate: 0.050107758620689655
target_rate: 0.05
N: 1856
decision: 0

Result: Fail to reject H0 (Model is acceptable)
